In [1]:
! pip install ddeint 
! pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Zenbook\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Zenbook\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ddeint import ddeint
from ipywidgets import interact, FloatSlider

def create_systems(beta, gamma, a, tau, mu_c, E_star, p, mu_f, eta, lam, mu_q):
    def dde_system(Y, t, Z):
        I, E, K, q = Y
        I_delayed, E_delayed, K_delayed, q_delayed = Z
        dI = beta * I - gamma * K * I
        dE = a * K_delayed * I_delayed - mu_c * (E - E_star)
        dK = p * E - mu_f * K - eta * gamma * I * K
        dq = lam * I - mu_q * q
        return [dI, dE, dK, dq]

    def ddeint_system(Y, t):
        I, E, K, q = Y(t)
        I_delayed, E_delayed, K_delayed, q_delayed = Y(t - tau)
        dI = beta * I - gamma * K * I
        dE = a * K_delayed * I_delayed - mu_c * (E - E_star)
        dK = p * E - mu_f * K - eta * gamma * I * K
        dq = lam * I - mu_q * q
        return [dI, dE, dK, dq]

    def solver_dde_rk2(t_span, dt, history, tau):
        t0, tf = t_span
        t_points = np.arange(t0, tf + dt, dt)
        n = len(t_points)
        L = int(round(tau / dt))
        sol = np.zeros((n, 4))
        sol[0] = history(0)

        for i in range(n - 1):
            t_curr = t_points[i]
            t_next = t_points[i+1]
            Y_curr = sol[i]

            def get_delayed(idx_shift):
                if idx_shift < 0:
                    return history(t_curr + idx_shift * dt)
                else:
                    return sol[idx_shift]

            Y_delayed_curr = get_delayed(i - L)
            f1 = dde_system(Y_curr, t_curr, Y_delayed_curr)
            Y_pred = Y_curr + dt * np.array(f1)

            Y_delayed_next = get_delayed(i + 1 - L)
            f2 = dde_system(Y_pred, t_next, Y_delayed_next)
            Y_next = Y_curr + (dt / 2.0) * (np.array(f1) + np.array(f2))

            sol[i+1] = Y_next
        return t_points, sol

    def verify_with_ddeint(t_span, dt):
        t_points = np.arange(t_span[0], t_span[1] + dt, dt)
        sol_ddeint = ddeint(ddeint_system, history, t_points)
        return t_points, sol_ddeint

    return solver_dde_rk2, verify_with_ddeint

def history(t):
    return np.array([0.001, 1.0, 0.0, 0.0])

def run_simulation(E_star, beta, gamma, a, tau, mu_c, p, mu_f, eta, lam, mu_q):
    solver_rk2, verify = create_systems(beta, gamma, a, tau, mu_c, E_star, p, mu_f, eta, lam, mu_q)
    t_span = (0, 100)
    dt = 0.1
    t_rk2, sol_rk2 = solver_rk2(t_span, dt, history, tau)
    t_ver, sol_ver = verify(t_span, dt)

    plt.figure(figsize=(12, 8))
    variables = ['I(t) - вирусы', 'E(t) - плазматические клетки', 'K(t) - антитела', 'q(t) - поражение органа']
    for i in range(4):
        plt.subplot(2, 2, i+1)
        plt.plot(t_rk2, sol_rk2[:, i], 'b-', label='RK2', linewidth=1.5)
        plt.plot(t_ver, sol_ver[:, i], 'r--', label='ddeint', linewidth=1.5)
        plt.title(variables[i])
        plt.xlabel('Время, сут')
        plt.ylabel('Концентрация / отн. ед.')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.show()
# уменьшать step для большей точности
interact(run_simulation,
         E_star=FloatSlider(min=1.0, max=5.0, step=0.5, value=0.5, description = 'Пороговый (нормальный) уровень плазматических клеток'),
         beta=FloatSlider(min=0.1, max=1.0, step=0.1, value=0.5, description = 'К. с-ти размножения вирусов'),
         gamma=FloatSlider(min=0.01, max=0.5, step=0.1, value=0.1, description = 'К. н-ции вируса антителами'),
         a=FloatSlider(min=0.05, max=0.5, step=0.1, value=0.2, description = 'К. стимуляции иммунной системы'),
         tau=FloatSlider(min=1.0, max=5.0, step=0.5, value=2.0, description = 'В. з-ки формирования иммунного ответа'),
         mu_c=FloatSlider(min=0.01, max=0.3, step=0.1, value=0.1, description = 'К. естественной убыли плазматических клеток'),
         p=FloatSlider(min=0.01, max=0.3, step=0.1, value=0.1, description = 'С. п-ва антител одной клеткой'),
         mu_f=FloatSlider(min=0.01, max=0.2, step=0.1, value=0.05, description = 'К. распада антител'),
         eta=FloatSlider(min=0.001, max=0.05, step=0.01, value=0.01, description = 'Р-д антител на нейт-цию одного вируса'),
         lam=FloatSlider(min=0.1, max=0.5, step=0.1, value=0.3, description = 'К. поражения органа вирусом'),
         mu_q=FloatSlider(min=0.01, max=0.2, step=0.1, value=0.05, description = ' Скорость восстановления органа'));

interactive(children=(FloatSlider(value=1.0, description='Пороговый (нормальный) уровень плазматических клеток…